# WordPiece: Subword Tokenization for BERT and Beyond

## What is WordPiece?

**WordPiece** is a subword tokenization algorithm developed by Google for Neural Machine Translation and later popularized by **BERT** (Bidirectional Encoder Representations from Transformers). It builds a vocabulary of subword units based on a language model that maximizes the probability of the training data.

**Key Paper**: [Google's Neural Machine Translation System: Bridging the Gap between Human and Machine Translation](https://arxiv.org/abs/1609.08144) - Wu et al., 2016

## WordPiece vs BPE: Key Differences

| Aspect | WordPiece | BPE |
|--------|-----------|-----|
| **Selection Criterion** | Likelihood-based (uses language model) | Frequency-based (most common pair) |
| **Starting Point** | Whole words | Individual characters |
| **Vocabulary** | Must include base tokens first | Can start from characters |
| **Algorithm** | Greedy merge based on perplexity | Frequent pattern merging |
| **Used By** | BERT, DistilBERT | GPT, GPT-2, RoBERTa |
|-------|-----------|-----|
| **Unknown Tokens** | Marked as [UNK] | May be split into subwords |

## How WordPiece Works

### Step-by-Step Process

**1. Initialize Vocabulary**
- Start with all individual characters in the corpus
- Add most common words as whole tokens

**2. Build Vocabulary Using Language Model**
- Iteratively add new subword units
- Each new unit is formed by joining two existing units
- Selection based on: **maximizes training data likelihood**

**3. Selection Formula**
```
Score = (freq of merged token) / (freq of token1 × freq of token2)
```
Higher score = better merge candidate

**4. Tokenization Process**
- For unknown words: greedily find longest subword in vocabulary
- Use dynamic programming for optimal segmentation

## Installation

In [ ]:
# Install required libraries
!pip install transformers tokenizers

## Method 1: Using Hugging Face Tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

# Create sample corpus
corpus = """
Natural language processing is a subfield of artificial intelligence.
It focuses on enabling computers to understand human language.
Deep learning has transformed the field of NLP significantly.
Transformer models have become the standard architecture for NLP.
BERT uses bidirectional encoding for better context understanding.
Tokenization breaks text into manageable pieces for models.
WordPiece is a subword tokenization algorithm developed by Google.
Pre-training and fine-tuning are common paradigms in modern NLP.
"""

# Save corpus to file
with open('corpus_wp.txt', 'w', encoding='utf-8') as f:
    f.write(corpus)

print("Corpus saved!")

In [ ]:
# Initialize tokenizer with WordPiece model
tokenizer = Tokenizer(WordPiece(unk_token='[UNK]'))

# Set pre-tokenizer (splits on whitespace first)
tokenizer.pre_tokenizer = Whitespace()

# Create trainer with WordPiece configuration
trainer = WordPieceTrainer(
    vocab_size=100,
    min_frequency=1,
    special_tokens=['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]'],
    continuing_subword_prefix='##',
    max_input_chars_per_word=100
)

# Train the tokenizer
tokenizer.train(files=['corpus_wp.txt'], trainer=trainer)

print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

In [ ]:
# Encode text
text = "Natural language processing is amazing"
output = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"Token IDs: {output.ids}")
print(f"Tokens: {output.tokens}")
print(f"Attention mask: {output.attention_mask}")

In [ ]:
# Decode back to text
decoded = tokenizer.decode(output.ids)
print(f"Decoded: {decoded}")

In [ ]:
# View vocabulary sample
vocab = tokenizer.get_vocab()
print("Sample vocabulary (first 30 items):")
for i, (token, idx) in enumerate(list(vocab.items())[:30]):
    print(f"  {token}: {idx}")

## Method 2: Using BERT Tokenizer (Pre-trained)

In [ ]:
from transformers import BertTokenizer

# Load pre-trained BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("BERT Tokenizer loaded!")
print(f"Vocabulary size: {bert_tokenizer.vocab_size}")

In [ ]:
# Encode text using BERT tokenizer
text = "WordPiece tokenization is used in BERT!"
tokens = bert_tokenizer.tokenize(text)
ids = bert_tokenizer.encode(text)
decoded = bert_tokenizer.decode(ids)

print(f"Original text: {text}")
print(f"WordPiece tokens: {tokens}")
print(f"Token IDs: {ids}")
print(f"Decoded: {decoded}")

In [ ]:
# Note the '##' prefix for subwords
text = "Subword tokenization handles unknown words well"
tokens = bert_tokenizer.tokenize(text)
print(f"Text: {text}")
print(f"Tokens: {tokens}")
print("\nNotice '##' prefix indicates continuation of a word")

## Understanding WordPiece Tokenization in Detail

In [ ]:
# Visualize how WordPiece handles different types of words
test_words = [
    "unbreakable",
    "internationalization",
    "machine",
    "learning",
    "artificial",
    "intelligence",
    "hello",
    "world"
]

print("WordPiece Tokenization Examples:")
print("-" * 50)
for word in test_words:
    tokens = bert_tokenizer.tokenize(word)
    print(f"{word:25} -> {tokens}")

In [ ]:
# Compare WordPiece with character-level and word-level
text = "deep learning models"

print(f"Text: '{text}'")
print(f"\nWordPiece tokens: {bert_tokenizer.tokenize(text)}")
print(f"\nCharacter-level: {list(text)}")

## BERT's Special Tokens

In [ ]:
print("BERT Special Tokens:")
print(f"  [PAD]:    {bert_tokenizer.pad_token_id}")
print(f"  [UNK]:    {bert_tokenizer.unk_token_id}")
print(f"  [CLS]:    {bert_tokenizer.cls_token_id}")
print(f"  [SEP]:    {bert_tokenizer.sep_token_id}")
print(f"  [MASK]:   {bert_tokenizer.mask_token_id}")

In [ ]:
# Full sentence encoding with BERT
text1 = "Natural language processing enables computers to understand text."
text2 = "It uses machine learning techniques."

encoded = bert_tokenizer(
    text1,
    text2,
    padding='max_length',
    truncation=True,
    max_length=20,
    return_tensors='pt'
)

print("Sentence pair encoding:")
print(f"Input IDs:\n{encoded['input_ids']}")
print(f"\nToken type IDs:\n{encoded['token_type_ids']}")
print(f"\nAttention mask:\n{encoded['attention_mask']}")

## WordPiece Algorithm Implementation (From Scratch)

In [ ]:
from collections import Counter
import re

class SimpleWordPiece:
    """
    Simplified WordPiece implementation to demonstrate the algorithm.
    This is NOT production-ready - use tokenizers or transformers library instead.
    """
    
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.unk_token = '[UNK]'
    
    def tokenize(self, text):
        """Tokenize text using greedy longest-match approach."""
        tokens = []
        start = 0
        
        while start < len(text):
            end = len(text)
            found = False
            
            while start < end:
                substr = text[start:end]
                
                if start > 0:
                    substr = '##' + substr
                
                if substr in self.vocab:
                    tokens.append(substr)
                    found = True
                    break
                
                end -= 1
            
            if not found:
                tokens.append(self.unk_token)
                end = start + 1
            
            start = end
        
        return tokens
    
    def train(self, corpus):
        """Train WordPiece vocabulary (simplified version)."""
        words = self._word_tokenize(corpus)
        word_counts = Counter(words)
        
        vocab = set()
        for word, count in word_counts.items():
            for c in word:
                vocab.add(c)
        vocab.add(self.unk_token)
        
        while len(vocab) < self.vocab_size:
            pairs = self._get_pair_counts(word_counts)
            if not pairs:
                break
            
            best_pair = max(pairs, key=pairs.get)
            merged = best_pair[0] + best_pair[1].lstrip('##')
            vocab.add(merged)
            
            word_counts = self._merge_pairs(word_counts, best_pair, merged)
        
        self.vocab = {v: i for i, v in enumerate(sorted(vocab))}
    
    def _word_tokenize(self, text):
        return re.findall(r'\S+', text.lower())
    
    def _get_pair_counts(self, word_counts):
        pairs = Counter()
        for word in word_counts:
            chars = list(word)
            for i in range(len(chars) - 1):
                pairs[(chars[i], chars[i+1])] += word_counts[word]
        return pairs
    
    def _merge_pairs(self, word_counts, pair, merged):
        new_counts = Counter()
        for word in word_counts:
            new_word = word.replace(''.join(pair), merged)
            new_counts[new_word] = word_counts[word]
        return new_counts

print("SimpleWordPiece class defined!")

In [ ]:
# Use our simplified WordPiece
wp = SimpleWordPiece(vocab_size=50)
wp.train(corpus)

test_word = "nlp"
tokens = wp.tokenize(test_word)
print(f"Tokenized '{test_word}': {tokens}")

## Models Using WordPiece

| Model | Tokenizer | Released By |
|-------|-----------|-------------|
| **BERT** | WordPiece | Google AI |
| **DistilBERT** | WordPiece | HuggingFace |
| **MobileBERT** | WordPiece | Google |
| **ALBERT** | SentencePiece | Google |
| **ERNIE** | WordPiece | Baidu |
| **XLM** | WordPiece | Facebook |
| **XLNet** | SentencePiece | Google/CMU |

## Advantages of WordPiece

### 1. Handles Out-of-Vocabulary (OOV) Words
- Unknown words are split into known subword units
- "unseen" → "un" + "##seen"

### 2. Preserves Word Boundaries
- '##' prefix marks continuation tokens
- Maintains readability of tokenized output

### 3. Language Model Based Selection
- Chooses merges that maximize training data likelihood
- Produces more linguistically meaningful subwords

### 4. Works Well with Morphologically Rich Languages
- German, Turkish, Finnish benefit from subword decomposition

### 5. Bidirectional Context (BERT)
- WordPiece enables efficient subword processing
- Both left-to-right and right-to-left attention

## Limitations of WordPiece

### 1. Greedy Tokenization
- May not produce globally optimal segmentation
- Only uses longest-match locally

### 2. Language-Specific Pretokenization
- Relies on whitespace tokenization first
- May not work well for languages without spaces (Chinese, Japanese)

### 3. Fixed Vocabulary
- Must define vocabulary size before training
- May need tuning for different corpora

### 4. Training Complexity
- Slower than BPE due to language model training
- Requires more computational resources

## WordPiece vs Other Tokenization Methods

In [ ]:
# Compare different tokenization methods
from transformers import BertTokenizer, GPT2Tokenizer, XLNetTokenizer

text = "Internationalization is important"

print("=" * 60)
print(f"Text: '{text}'")
print("=" * 60)

print(f"\nWordPiece (BERT):")
print(f"  Tokens: {bert_tokenizer.tokenize(text)}")

print(f"\nBPE (GPT-2):")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
print(f"  Tokens: {gpt2_tokenizer.tokenize(text)}")

## Practical Usage in BERT Pipeline

In [ ]:
# Complete BERT tokenization pipeline example
import torch
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Example: Sentiment classification input
texts = [
    "I love this movie, it's fantastic!",
    "This is terrible, I hate it.",
    "It's okay, not great not terrible."
]

print("BERT Tokenization Pipeline:")
print("-" * 50)

for text in texts:
    # Encode text
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=15,
        return_tensors='pt'
    )
    
    print(f"Text: {text}")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
    print(f"IDs: {encoded['input_ids'][0].tolist()}")
    print()

## Visual: Tokenization Process

In [ ]:
# Show step-by-step tokenization
def show_tokenization_steps(tokenizer, text):
    print(f"Original text: '{text}'")
    print(f"\nStep 1 - Original tokens (word-level):")
    print(f"  {text.split()}")
    
    print(f"\nStep 2 - WordPiece tokens:")
    wp_tokens = tokenizer.tokenize(text)
    print(f"  {wp_tokens}")
    
    print(f"\nStep 3 - Token IDs:")
    ids = tokenizer.encode(text)
    print(f"  {ids}")

show_tokenization_steps(bert_tokenizer, "The quick brown fox jumps")

## Summary

| Aspect | WordPiece |
|--------|-----------|
| **Type** | Subword tokenization |
| **Selection** | Likelihood-based (language model) |
| **Subword Prefix** | '##' for continuation |
| **Key Models** | BERT, DistilBERT, ERNIE |
| **OOV Handling** | Split into known subwords |
| **Advantage** | Linguistically meaningful splits |
| **Limitation** | Greedy tokenization, language-specific |

## Key Takeaways

1. **WordPiece** is Google's subword algorithm used primarily in BERT

2. **Selection criterion** is based on maximizing training data likelihood

3. **'##' prefix** indicates tokens that are part of a word

4. **Handles OOV** by breaking unknown words into known subwords

5. **Different from BPE** - uses language model vs frequency-based selection

6. **Pre-trained tokenizers** from HuggingFace are production-ready